# 03 Rule Classification

Apply deterministic first-pass rules to the latest inventory output and produce a review table. Still dry-run only.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

POLICY_PATH = PROJECT_ROOT / 'policy' / 'SCH_fileserver_policy_v2_5.yaml'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('POLICY_PATH =', POLICY_PATH)


PROJECT_ROOT = c:\00_dev\SCH-FILE-ORGANIZER
POLICY_PATH = c:\00_dev\SCH-FILE-ORGANIZER\policy\SCH_fileserver_policy_v2_5.yaml


In [2]:
from datetime import datetime
import pandas as pd

from src.policy_loader import PolicyLoader
from src.rules import classify_inventory, save_rule_outputs

policy = PolicyLoader.from_file(POLICY_PATH)
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
inventory_files = list(OUTPUT_DIR.glob('inventory_*.parquet'))
assert inventory_files, 'No inventory parquet files found. Run 02_inventory.ipynb first.'

# Pick the newest file by filesystem timestamp, not filename order.
latest_inventory = max(inventory_files, key=lambda p: p.stat().st_mtime)
print('Using inventory:', latest_inventory.name)
inv = pd.read_parquet(latest_inventory)
print('Rows:', len(inv))
print('Columns:', list(inv.columns))


Using inventory: inventory_HTL0049-01_OITYLO-KOKKALA_MANI_20260309_075647.parquet
Rows: 7552
Columns: ['scan_root', 'absolute_path', 'relative_path', 'parent_relative', 'filename', 'stem', 'suffix', 'size_bytes', 'modified_at', 'created_at', 'depth_segments', 'path_length', 'filename_length', 'is_hidden', 'is_symlink', 'top_segment', 'hash', 'is_duplicate_hash', 'duplicate_group_size']


## Schema note
`classify_inventory()` now backfills missing inventory fields such as `filename`, `suffix`, `parent_relative`, `path_length`, and `filename_length` if you loaded an older inventory parquet. For best consistency, rerun `02_inventory.ipynb` after policy or scanner changes.


In [4]:
classified = classify_inventory(inv, POLICY_PATH)
classified[['relative_path', 'rule_status', 'rule_reason', 'rule_confidence', 'proposed_relative_target']].head(5)

,relative_path,rule_status,rule_reason,rule_confidence,proposed_relative_target
0,01_DEVELOPMENT\ΠΡΟΫΠΟΛΟΓΙΣΜΟΣ ΕΡΓΟΥ.docx,review,filename_not_in_canonical_pattern,low,
1,02_LAND_ACQUISITION\2021.1.8_ ΙΔΙΩΤΙΚΟ ΣΥΜΦΩΝΗ...,review,filename_not_in_canonical_pattern,low,
2,02_LAND_ACQUISITION\2021.10.19 Πρακτικο Γ.Σ. Π...,review,filename_not_in_canonical_pattern,low,
3,02_LAND_ACQUISITION\2021.10.19 Πρακτικό Αγορά...,review,filename_not_in_canonical_pattern,low,
4,02_LAND_ACQUISITION\Taxis NET -Εικόνα Επαγγελμ...,review,filename_not_in_canonical_pattern,low,


In [5]:
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_base = OUTPUT_DIR / f'rule_classification_{stamp}'
csv_path, parquet_path = save_rule_outputs(classified, output_base)
print('CSV:', csv_path)
print('Parquet:', parquet_path)


CSV: c:\00_dev\SCH-FILE-ORGANIZER\data\outputs\rule_classification_20260309_082249\rule_classification_20260309_082249.csv
Parquet: c:\00_dev\SCH-FILE-ORGANIZER\data\outputs\rule_classification_20260309_082249\rule_classification_20260309_082249.parquet


In [6]:
classified.groupby('rule_status').size().sort_values(ascending=False).to_frame('count')

,count
rule_status,
move_to_special_folder,7149
archive_or_delete_candidate,255
review,148


In [7]:
classified[classified['rule_status'] == 'archive_or_delete_candidate'][['relative_path', 'filename', 'rule_reason', 'proposed_relative_target']].head(10)

,relative_path,filename,rule_reason,proposed_relative_target
121,03_PERMITTING\Α.Α 383786\ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,21011_YDREYSH_PLANS.dwl,junk_system_or_temp_file,
122,03_PERMITTING\Α.Α 383786\ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,21011_YDREYSH_PLANS.dwl2,junk_system_or_temp_file,
126,03_PERMITTING\Α.Α 383786\ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,Thumbs.db,junk_system_or_temp_file,
353,04_DESIGN_ENGINEERING\ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ\24...,01_SONADO_OITYLO_KATOPSH A STATHMIS.bak,junk_system_or_temp_file,
355,04_DESIGN_ENGINEERING\ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ\24...,02_SONADO_OITYLO_KATOPSH B STATHMIS.bak,junk_system_or_temp_file,
357,04_DESIGN_ENGINEERING\ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ\24...,03_SONADO_OITYLO_KATOPSH C STATHMIS.bak,junk_system_or_temp_file,
359,04_DESIGN_ENGINEERING\ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ\24...,04_SONADO_OITYLO_KATOPSH D STATHMIS.bak,junk_system_or_temp_file,
542,04_DESIGN_ENGINEERING\ΔΙΑΦΟΡΑ\ΜΕΛΕΤΕΣ-ΕΦΑΡΜΟΓΗ...,01_SONADO_OITYLO_KATOPSH A STATHMIS.bak,junk_system_or_temp_file,
544,04_DESIGN_ENGINEERING\ΔΙΑΦΟΡΑ\ΜΕΛΕΤΕΣ-ΕΦΑΡΜΟΓΗ...,02_SONADO_OITYLO_KATOPSH B STATHMIS.bak,junk_system_or_temp_file,
546,04_DESIGN_ENGINEERING\ΔΙΑΦΟΡΑ\ΜΕΛΕΤΕΣ-ΕΦΑΡΜΟΓΗ...,03_SONADO_OITYLO_KATOPSH C STATHMIS.bak,junk_system_or_temp_file,


In [8]:
cols = [c for c in ['relative_path', 'filename', 'rule_reason', 'special_folder_target', 'proposed_relative_target'] if c in classified.columns]
classified[classified['rule_status'] == 'move_to_special_folder'][cols].head(10)

,relative_path,filename,rule_reason,special_folder_target,proposed_relative_target
8,02_LAND_ACQUISITION\ΤΙΤΛΟΙ ΑΚΙΝΗΤΟΥ\20210108 Ι...,20210108 ΙΔΙΩΤΙΚΟ ΣΥΜΦΩΝΗΤΙΚΟ ΕΠΑΓΓΕΛΜΑΤΙΚΗΣ Μ...,duplicate_exact_hash,_DUPLICATED,
15,03_PERMITTING\IKA -SONADO_.pdf,IKA -SONADO_.pdf,duplicate_exact_hash,_DUPLICATED,
16,03_PERMITTING\Α.Α 364747\01. ΤΟΠΟΓΡΑΦΙΚΑ\01_SO...,01_SONADO IKE_TOPOGRAFIKO DIAGRAMMA.pdf,duplicate_exact_hash,_DUPLICATED,
17,03_PERMITTING\Α.Α 383786\ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,01_SONADO IKE_TEXNIKH EKTHESH_SA.pdf,duplicate_exact_hash,_DUPLICATED,
18,03_PERMITTING\Α.Α 383786\ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,01_SONADO_OITYLO_KATOPSH ST A'.pdf,duplicate_exact_hash,_DUPLICATED,
19,03_PERMITTING\Α.Α 383786\ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,02_SONADO IKE_TOPOGRAFIKO DIAGRAMMA_SA.pdf,duplicate_exact_hash,_DUPLICATED,
20,03_PERMITTING\Α.Α 383786\ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,02_SONADO_OITYLO_KATOPSH ST B'.pdf,duplicate_exact_hash,_DUPLICATED,
21,03_PERMITTING\Α.Α 383786\ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,03_SONADO IKE_DIAGRAMMA DOMHSHS_SA.pdf,duplicate_exact_hash,_DUPLICATED,
22,03_PERMITTING\Α.Α 383786\ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,03_SONADO_OITYLO_KATOPSH ST C'.pdf,duplicate_exact_hash,_DUPLICATED,
23,03_PERMITTING\Α.Α 383786\ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,04_SONADO_OITYLO_KATOPSH ST D'.pdf,duplicate_exact_hash,_DUPLICATED,


In [9]:
classified[classified['rule_status'] == 'compliant_keep_review_path'][['relative_path', 'filename', 'parsed_phase', 'parsed_doc_type', 'default_folder_subpath', 'proposed_relative_target']].head(10)

,relative_path,filename,parsed_phase,parsed_doc_type,default_folder_subpath,proposed_relative_target


In [10]:
classified[classified['rule_status'] == 'review'][['relative_path', 'filename', 'rule_reason', 'path_risk', 'filename_risk']].head(10)

,relative_path,filename,rule_reason,path_risk,filename_risk
0,01_DEVELOPMENT\ΠΡΟΫΠΟΛΟΓΙΣΜΟΣ ΕΡΓΟΥ.docx,ΠΡΟΫΠΟΛΟΓΙΣΜΟΣ ΕΡΓΟΥ.docx,filename_not_in_canonical_pattern,,
1,02_LAND_ACQUISITION\2021.1.8_ ΙΔΙΩΤΙΚΟ ΣΥΜΦΩΝΗ...,2021.1.8_ ΙΔΙΩΤΙΚΟ ΣΥΜΦΩΝΗΤΙΚΟ ΕΠΑΓΓΕΛΜΑΤΙΚΗΣ ...,filename_not_in_canonical_pattern,,
2,02_LAND_ACQUISITION\2021.10.19 Πρακτικο Γ.Σ. Π...,2021.10.19 Πρακτικο Γ.Σ. Πώλησης αγροτεμαχίου ...,filename_not_in_canonical_pattern,,
3,02_LAND_ACQUISITION\2021.10.19 Πρακτικό Αγορά...,2021.10.19 Πρακτικό Αγοράς αγροτεμαχίου από ...,filename_not_in_canonical_pattern,,
4,02_LAND_ACQUISITION\Taxis NET -Εικόνα Επαγγελμ...,Taxis NET -Εικόνα Επαγγελματικής Μίσθωσης NYCO...,filename_not_in_canonical_pattern,,
5,02_LAND_ACQUISITION\ΚΤΗΜΑΤΟΛΟΓΙΟ\Apospasma3011...,Apospasma301102601181_0_0.pdf,filename_not_in_canonical_pattern,,
6,02_LAND_ACQUISITION\ΚΤΗΜΑΤΟΛΟΓΙΟ\KD30110260118...,KD301102601181_0_0.pdf,filename_not_in_canonical_pattern,,
7,02_LAND_ACQUISITION\ΤΙΤΛΟΙ ΑΚΙΝΗΤΟΥ\20160305_6...,20160305_6259 ΣΥΜΒΟΛΑΙΟ NYCONTEC M ΕΠΕ_ΓΙΑΝΝΑΡ...,filename_not_in_canonical_pattern,,
9,02_LAND_ACQUISITION\ΤΙΤΛΟΙ ΑΚΙΝΗΤΟΥ\20211209_7...,20211209_7664_NYCONTEC-SONADO_ΠΩΛΗΤΗΡΙΟ&ΣΥΣΤΑΣ...,filename_not_in_canonical_pattern,,
10,02_LAND_ACQUISITION\ΤΙΤΛΟΙ ΑΚΙΝΗΤΟΥ\7664. 9-12...,7664. 9-12-2021 Συμβόλαιο Πιστοποιητικό Ιδιωκτ...,filename_not_in_canonical_pattern,,


In [11]:
classified.sort_values(['action_priority', 'relative_path']).head(10)

,scan_root,absolute_path,relative_path,parent_relative,filename,stem,suffix,size_bytes,modified_at,created_at,...,routing_basis,counterparty_rule_ok,counterparty_rule_reason,default_folder_subpath,current_folder_subpath,path_length_warning,filename_length_warning,path_risk,filename_risk,action_priority
121,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,03_PERMITTING\Α.Α 383786\ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,03_PERMITTING/Α.Α 383786/ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,21011_YDREYSH_PLANS.dwl,21011_YDREYSH_PLANS,.dwl,51,2023-03-01 14:05:28.639566660,2026-03-09 05:53:20.530429840,...,,False,not_applicable,,03_PERMITTING/Α.Α 383786/ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,False,False,,,10
122,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,03_PERMITTING\Α.Α 383786\ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,03_PERMITTING/Α.Α 383786/ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,21011_YDREYSH_PLANS.dwl2,21011_YDREYSH_PLANS,.dwl2,217,2023-03-01 14:05:28.920873404,2026-03-09 05:53:20.537228107,...,,False,not_applicable,,03_PERMITTING/Α.Α 383786/ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,False,False,,,10
126,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,03_PERMITTING\Α.Α 383786\ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,03_PERMITTING/Α.Α 383786/ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,Thumbs.db,Thumbs,.db,6144,2023-02-13 12:22:05.263608456,2026-03-09 05:53:20.565987349,...,,False,not_applicable,,03_PERMITTING/Α.Α 383786/ΕΓΚΕΚΡΙΜΕΝΑ ΣΧΕΔΙΑ ΜΕ...,False,False,,,10
353,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,04_DESIGN_ENGINEERING\ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ\24...,04_DESIGN_ENGINEERING/ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ/24...,01_SONADO_OITYLO_KATOPSH A STATHMIS.bak,01_SONADO_OITYLO_KATOPSH A STATHMIS,.bak,664636,2022-11-02 15:31:59.000000000,2026-03-09 05:53:22.279737949,...,,False,not_applicable,,04_DESIGN_ENGINEERING/ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ/24...,False,False,,,10
355,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,04_DESIGN_ENGINEERING\ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ\24...,04_DESIGN_ENGINEERING/ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ/24...,02_SONADO_OITYLO_KATOPSH B STATHMIS.bak,02_SONADO_OITYLO_KATOPSH B STATHMIS,.bak,9554920,2022-11-02 15:31:59.000000000,2026-03-09 05:53:22.290498972,...,,False,not_applicable,,04_DESIGN_ENGINEERING/ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ/24...,False,False,,,10
357,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,04_DESIGN_ENGINEERING\ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ\24...,04_DESIGN_ENGINEERING/ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ/24...,03_SONADO_OITYLO_KATOPSH C STATHMIS.bak,03_SONADO_OITYLO_KATOPSH C STATHMIS,.bak,26347123,2022-11-02 15:31:59.000000000,2026-03-09 05:53:22.307997704,...,,False,not_applicable,,04_DESIGN_ENGINEERING/ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ/24...,False,False,,,10
359,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,04_DESIGN_ENGINEERING\ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ\24...,04_DESIGN_ENGINEERING/ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ/24...,04_SONADO_OITYLO_KATOPSH D STATHMIS.bak,04_SONADO_OITYLO_KATOPSH D STATHMIS,.bak,14532003,2022-11-02 15:31:59.000000000,2026-03-09 05:53:22.335934401,...,,False,not_applicable,,04_DESIGN_ENGINEERING/ΑΡΧΙΤΕΚΤΟΝΙΚΑ- ΓΕΝΙΚΑ/24...,False,False,,,10
542,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,04_DESIGN_ENGINEERING\ΔΙΑΦΟΡΑ\ΜΕΛΕΤΕΣ-ΕΦΑΡΜΟΓΗ...,04_DESIGN_ENGINEERING/ΔΙΑΦΟΡΑ/ΜΕΛΕΤΕΣ-ΕΦΑΡΜΟΓΗ...,01_SONADO_OITYLO_KATOPSH A STATHMIS.bak,01_SONADO_OITYLO_KATOPSH A STATHMIS,.bak,664636,2022-11-02 15:31:59.000000000,2026-03-09 05:53:23.451033354,...,,False,not_applicable,,04_DESIGN_ENGINEERING/ΔΙΑΦΟΡΑ/ΜΕΛΕΤΕΣ-ΕΦΑΡΜΟΓΗ...,False,False,,,10
544,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,C:\Users\User\Desktop\HTL0049-01_OITYLO-KOKKAL...,04_DESIGN_ENGINEERING\ΔΙΑΦΟΡΑ\ΜΕΛΕΤΕΣ-ΕΦΑΡΜΟΓΗ...,04_DESIGN_ENGINEERING/ΔΙΑΦΟΡΑ/ΜΕΛΕΤΕΣ-ΕΦΑΡΜΟΓΗ...,02_SONADO_OITYLO_KATOPSH B STATHMIS.bak,02_SONADO_